# Video 5: Tools in CrewAI

## What We Will Build
In this notebook, we will build **custom CrewAI tools** and assign them to agents. You will learn:
- Why agents need tools
- How to create custom tools using the `@tool` decorator
- How to assign tools to agents
- How agents execute tool calls
- Error handling for tools
- Best practices for tool design

By the end, you will have agents that can calculate, read files, and call APIs.

## 1. Install & Import

In [ ]:
!pip install -q crewai crewai-tools langchain-openai python-dotenv

## 2. Imports & Setup

In [ ]:
from crewai import Agent, Task, Crew, Processfrom crewai.tools import toolfrom langchain_openai import ChatOpenAIimport osimport jsonfrom dotenv import load_dotenvload_dotenv()llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=os.getenv('OPENAI_API_KEY'))print('Setup complete')

## 3. Theory: Tools in CrewAI

### Why Agents Need Tools
Tools extend an agent's capabilities beyond text generation:
- **Calculations**: Math, statistics, data processing
- **Data access**: Read files, query databases, call APIs
- **External actions**: Send emails, create tickets, post updates
- **Search**: Web search, document retrieval, knowledge lookup

### Tool Design Principles
1. **Clear description**: The agent must understand what the tool does
2. **Type hints**: Help the LLM understand parameters
3. **Error handling**: Graceful failures prevent workflow crashes
4. **Focused scope**: One tool = one clear purpose

### Creating Custom Tools
Use the `@tool` decorator from `crewai.tools`:
```python
@tool
def my_tool(param: str) -> str:
    """Clear description of what this tool does."""
    # Implementation
    return result
```

## 4. Create a Calculator Tool

In [ ]:
@tooldef calculator(expression: str) -> str:    """Evaluate a mathematical expression safely.        Args:        expression: A mathematical expression like '2 + 2' or '(10 * 5) / 2'    """    try:        allowed_names = {            'abs': abs, 'round': round, 'max': max, 'min': min,            'sum': sum, 'pow': pow, 'len': len        }        result = eval(expression, {'__builtins__': {}}, allowed_names)        return f'Result: {result}'    except Exception as e:        return f'Error: {str(e)}. Please provide a valid mathematical expression.'print('Calculator tool created')print(f'  Name: {calculator.name}')print(f'  Description: {calculator.description}')

## 5. Create a File Reading Tool

In [ ]:
@tooldef read_file_content(file_path: str) -> str:    """Read and return the content of a text file.        Args:        file_path: The path to the file to read    """    try:        with open(file_path, 'r', encoding='utf-8') as f:            content = f.read()        return f'File content:\n{content}'    except FileNotFoundError:        return f'Error: File not found at {file_path}'    except Exception as e:        return f'Error reading file: {str(e)}'print('File reading tool created')

## 6. Create a Mock API Tool

In [ ]:
@tooldef fetch_weather_data(city: str) -> str:    """Fetch weather data for a given city (mock API).        Args:        city: The name of the city to get weather for    """    weather_db = {        'new york': {'temp': 22, 'condition': 'Sunny', 'humidity': 45},        'london': {'temp': 15, 'condition': 'Rainy', 'humidity': 80},        'tokyo': {'temp': 20, 'condition': 'Cloudy', 'humidity': 60},        'paris': {'temp': 18, 'condition': 'Partly Cloudy', 'humidity': 55},        'sydney': {'temp': 28, 'condition': 'Hot', 'humidity': 40},    }        city_lower = city.lower().strip()    if city_lower in weather_db:        data = weather_db[city_lower]        return (            f'Weather for {city.title()}:\n'            f'  Temperature: {data["temp"]}°C\n'            f'  Condition: {data["condition"]}\n'            f'  Humidity: {data["humidity"]}%'        )    else:        available = ', '.join(weather_db.keys())        return f'Weather data not available for {city}. Available cities: {available}'print('Weather API tool created')

## 7. Create a Data Lookup Tool

In [ ]:
@tooldef search_company_info(company_name: str) -> str:    """Search for company information (mock database).        Args:        company_name: The name of the company to search for    """    companies = {        'openai': {            'founded': 2015,            'ceo': 'Sam Altman',            'headquarters': 'San Francisco, CA',            'products': 'ChatGPT, GPT-4, DALL-E, API'        },        'google': {            'founded': 1998,            'ceo': 'Sundar Pichai',            'headquarters': 'Mountain View, CA',            'products': 'Search, Cloud, Android, Gemini'        },        'microsoft': {            'founded': 1975,            'ceo': 'Satya Nadella',            'headquarters': 'Redmond, WA',            'products': 'Windows, Office, Azure, Copilot'        },    }        name_lower = company_name.lower().strip()    if name_lower in companies:        info = companies[name_lower]        return (            f'Company: {company_name.title()}\n'            f'Founded: {info["founded"]}\n'            f'CEO: {info["ceo"]}\n'            f'Headquarters: {info["headquarters"]}\n'            f'Products: {info["products"]}'        )    else:        available = ', '.join(companies.keys())        return f'Company not found. Available: {available}'print('Company lookup tool created')

## 8. Create a Test File

In [ ]:
# Create a sample file for the file reading toolsample_content = """Project Update - Q1 2025=========================Team: AI Research DivisionStatus: On TrackKey Achievements:1. Launched new LLM evaluation framework2. Reduced inference latency by 40%3. Published 3 research papersNext Steps:- Deploy multi-agent system to production- Expand training dataset by 2x"""with open('sample_project.txt', 'w') as f:    f.write(sample_content)print('Sample file created: sample_project.txt')

## 9. Create Agents with Tools

In [ ]:
# Create agents with specific toolsmath_agent = Agent(    role='Mathematical Analyst',    goal='Perform accurate mathematical calculations and data analysis',    backstory=(        'You are a precise mathematical analyst who uses calculation tools to ensure accuracy. '        'You never guess - you always verify with calculations.'    ),    llm=llm,    tools=[calculator],    verbose=True,    allow_delegation=False)data_agent = Agent(    role='Data Researcher',    goal='Gather accurate data from files, APIs, and databases',    backstory=(        'You are a data researcher who excels at finding and extracting information from '        'various sources including files, APIs, and databases.'    ),    llm=llm,    tools=[read_file_content, fetch_weather_data, search_company_info],    verbose=True,    allow_delegation=False)print('Agents created with tools assigned')print(f'  Math Agent tools: {[t.name for t in math_agent.tools]}')print(f'  Data Agent tools: {[t.name for t in data_agent.tools]}')

## 10. Create Tasks and Execute

In [ ]:
# Task 1: Math calculationmath_task = Task(    description=(        'Calculate the following: A company has 120 employees. Each employee works 8 hours a day. '        'The average hourly rate is $45. What is the total daily payroll cost? '        'Then calculate the monthly cost assuming 22 working days per month.'    ),    expected_output='Accurate calculation of daily and monthly payroll costs.',    agent=math_agent)# Task 2: Data gatheringdata_task = Task(    description=(        '1. Read the file sample_project.txt and summarize its contents.\n'        '2. Get the weather data for London and Tokyo.\n'        '3. Search for information about OpenAI.\n'        'Compile all findings into a single report.'    ),    expected_output='A comprehensive report with file summary, weather data, and company info.',    agent=data_agent)# Create crewtool_crew = Crew(    agents=[math_agent, data_agent],    tasks=[math_task, data_task],    process=Process.sequential,    verbose=True)print('Crew ready with tool-enabled agents')

## 11. Execute the Tool-Enabled Crew

In [ ]:
print('\n' + '=' * 60)print('EXECUTING TOOL-ENABLED CREW')print('=' * 60)result = tool_crew.kickoff()print('\n' + '=' * 60)print('FINAL OUTPUT')print('=' * 60)print(result)

## 12. Tool Error Handling

Notice how our tools handle errors gracefully:
- Invalid expressions return helpful error messages
- Missing files are reported clearly
- Unknown cities/companies suggest alternatives

This prevents the agent from crashing and helps it recover.

## 13. Tool Security Best Practices

1. **Validate inputs** — Sanitize parameters before processing
2. **Limit scope** — Tools should do one thing well
3. **Handle errors** — Never let exceptions crash the workflow
4. **Log actions** — Track what tools are called with what args
5. **Rate limit** — Prevent abuse of expensive APIs
6. **Access control** — Restrict sensitive tool access

---

## ✅ Summary

You have successfully:
1. ✅ Created custom tools with `@tool` decorator
2. ✅ Built a calculator, file reader, weather API, and company lookup tool
3. ✅ Assigned tools to specialized agents
4. ✅ Executed tool calls through the crew workflow
5. ✅ Observed how agents use tools to complete tasks
6. ✅ Implemented error handling in tools

In the next notebook, we will explore **Memory and Knowledge** in CrewAI.